<a href="https://colab.research.google.com/github/peteparker123/deep_agent/blob/main/deep_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU deepagents langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.1/236.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 25.2 MB/s eta 0:00:00


In [ ]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 79.5 MB/s eta 0:00:00


In [ ]:
import os
from typing import Literal

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchResults
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

# Store your key in an environment variable, not hardcoded in the file
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    api_key="your gemini api key",
)

# Initialize the search tool
search = DuckDuckGoSearchResults()


def internet_search(
    query: str,
    max_results: int = 2,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return search.invoke(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


# System prompt — now tells the agent to save its report to a file itself
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

Use `internet_search` at most 2 times total for this task. Do not repeat
similar queries. Once you have enough information, stop searching and write
the report directly.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.

## Saving your report

Once your research is complete, use the `write_file` tool to save your final
polished report to a file named `report.md`. Write the full report as plain
text/markdown content — this is the deliverable the user will read.
"""

# FilesystemBackend gives the agent a write(file_path, content) tool
# backed by real disk I/O under root_dir. virtual_mode=True sandboxes
# paths so the agent can't escape root_dir with ../ or absolute paths.
agent = create_deep_agent(
    model=llm,
    tools=[internet_search],
    system_prompt=research_instructions,
    backend=FilesystemBackend(root_dir=".", virtual_mode=True),
)

for event in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is LangGraph? Save your final report to report.md.",
            }
        ]
    },
    stream_mode="updates",
):
    for node, value in event.items():
        print(f"\n===== {node} =====")

        if value is None:
            continue

        if "messages" in value:
            for msg in value["messages"]:
                print(type(msg).__name__)

                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    print("Tool Calls:", msg.tool_calls)

                print("Content:", msg.content)

print("\n✅ Done. Check report.md in the current directory for the saved report.")


===== PatchToolCallsMiddleware.before_agent =====

===== model =====
AIMessage
Tool Calls: [{'name': 'internet_search', 'args': {'query': 'What is LangGraph key features architecture LangChain'}, 'id': '8brd6j1j', 'type': 'tool_call'}]
Content: []

===== TodoListMiddleware.after_model =====

===== tools =====
ToolMessage
Content: snippet: LangGraph is an open-source framework from LangChain designed to build and manage AI agent workflows using graph-based structures. It allows developers to define workflows as nodes and edges, making complex agent interactions more structured, scalable and easier to control.·, title: What is LangGraph - GeeksforGeeks, link: https://www.geeksforgeeks.org/machine-learning/what-is-langgraph/, snippet: LangChain and LangGraph are tools that help build AI workflows. LangChain works in a step by step way, while LangGraph allows more flexible paths. Both make it easier to connect AI models with data and tools to create smarter applications., title: LangChain